In [10]:
import random
import numpy as np

In [11]:
# ─────────────────────────────────────────────────────────────
# DATA STRUCTURES
# ─────────────────────────────────────────────────────────────
# Problem instance is represented as:
#   n          : number of items
#   m          : number of constraints
#   values     : list of length n, values[j] = value of item j
#   weights    : 2D list, weights[j][i] = weight of item j under constraint i
#   capacities : list of length m, capacities[i] = capacity of constraint i

# A solution is a list of n bits, e.g. [1, 0, 1, 0, 1]
# solution[j] = 1 means item j is included


# ─────────────────────────────────────────────────────────────
# STEP 1 — EFFICIENCY SCORES (computed once, never changes)
# ─────────────────────────────────────────────────────────────
def compute_efficiency_scores(values, weights, capacities, n, m):
    """
    Chu & Beasley efficiency score for each item:
    u_j = v_j / sum over i of (w_ji / W_i)

    Higher score = more value per unit of normalized resource consumed.
    Used by repair operator to decide which items to drop or add.
    Computed once at the start and reused throughout the entire run.
    """
    scores = []
    for j in range(n):
        denominator = sum(weights[j][i] / capacities[i] for i in range(m))
        if denominator == 0:
            scores.append(float('inf'))
        else:
            scores.append(values[j] / denominator)
    return scores


# ─────────────────────────────────────────────────────────────
# STEP 2 — FEASIBILITY CHECK
# ─────────────────────────────────────────────────────────────
def is_feasible(solution, weights, capacities, n, m):
    """
    Returns True if solution satisfies all m capacity constraints.
    """
    for i in range(m):
        total = sum(weights[j][i] * solution[j] for j in range(n))
        if total > capacities[i]:
            return False
    return True


# ─────────────────────────────────────────────────────────────
# STEP 3 — REPAIR OPERATOR
# ─────────────────────────────────────────────────────────────
def repair(solution, values, weights, capacities, efficiency_scores, n, m):
    """
    Chu & Beasley repair operator. Two phases:

    Phase 1 — Fix infeasibility:
        While any constraint is violated, remove the included item
        with the LOWEST efficiency score. Keep removing until feasible.

    Phase 2 — Improve solution:
        Try adding excluded items in order of HIGHEST efficiency score.
        Add an item only if it keeps all constraints feasible.
    """
    solution = solution.copy()

    # sort items by efficiency score ascending (lowest first for dropping)
    items_by_efficiency_asc  = sorted(range(n), key=lambda j: efficiency_scores[j])
    # sort items by efficiency score descending (highest first for adding)
    items_by_efficiency_desc = sorted(range(n), key=lambda j: efficiency_scores[j],
                                      reverse=True)

    # Phase 1 — drop lowest efficiency items until feasible
    while not is_feasible(solution, weights, capacities, n, m):
        for j in items_by_efficiency_asc:
            if solution[j] == 1:
                solution[j] = 0
                break  # drop one item then recheck feasibility

    # Phase 2 — greedily add highest efficiency items that still fit
    for j in items_by_efficiency_desc:
        if solution[j] == 0:
            solution[j] = 1
            if not is_feasible(solution, weights, capacities, n, m):
                solution[j] = 0  # does not fit, put it back

    return solution


# ─────────────────────────────────────────────────────────────
# STEP 4 — FITNESS FUNCTION
# ─────────────────────────────────────────────────────────────
def fitness(solution, values, n):
    """
    Total value of items included in solution.
    Higher is better.
    Solution must be feasible before calling this —
    repair operator guarantees feasibility.
    """
    return sum(values[j] * solution[j] for j in range(n))


# ─────────────────────────────────────────────────────────────
# STEP 5 — INITIALIZATION (uniform random, Chu & Beasley style)
# ─────────────────────────────────────────────────────────────
def initialize_population(population_size, n):
    """
    Generate population_size random binary strings.
    Each bit set to 1 with probability 0.5 independently.
    This is the standard Chu & Beasley initialization.
    Solutions are likely infeasible — repair operator fixes them.
    """
    return [[random.randint(0, 1) for _ in range(n)]
            for _ in range(population_size)]


# ─────────────────────────────────────────────────────────────
# STEP 6 — SELECTION (binary tournament)
# ─────────────────────────────────────────────────────────────
def tournament_select(population, fitness_scores):
    """
    Pick two individuals at random.
    Return the one with higher fitness.
    Simple, effective, no bias toward extreme fitness values.
    """
    a = random.randint(0, len(population) - 1)
    b = random.randint(0, len(population) - 1)
    if fitness_scores[a] >= fitness_scores[b]:
        return population[a]
    else:
        return population[b]


# ─────────────────────────────────────────────────────────────
# STEP 7 — CROSSOVER (uniform)
# ─────────────────────────────────────────────────────────────
def uniform_crossover(parent1, parent2, n):
    """
    For each bit position independently flip a fair coin.
    Heads: child gets parent1's bit.
    Tails: child gets parent2's bit.
    More disruptive than single point but explores more of the space.
    """
    child = []
    for j in range(n):
        if random.random() < 0.5:
            child.append(parent1[j])
        else:
            child.append(parent2[j])
    return child


# ─────────────────────────────────────────────────────────────
# STEP 8 — MUTATION (bit flip)
# ─────────────────────────────────────────────────────────────
def mutate(solution, n, mutation_rate=None):
    """
    Flip each bit independently with probability mutation_rate.
    Chu & Beasley use mutation_rate = 1/n
    meaning on average one bit flips per solution.
    """
    if mutation_rate is None:
        mutation_rate = 1.0 / n

    solution = solution.copy()
    for j in range(n):
        if random.random() < mutation_rate:
            solution[j] = 1 - solution[j]  # flip 0→1 or 1→0
    return solution


# ─────────────────────────────────────────────────────────────
# STEP 9 — THE FULL GA
# ─────────────────────────────────────────────────────────────
def chu_beasley_ga(n, m, values, weights, capacities,
                   population_size=200,
                   max_offspring=10000,
                   verbose=True):
    """
    Full Chu & Beasley steady-state GA for the MKP.

    Steady state means: generate one offspring at a time,
    insert it into population only if it beats the worst member.
    Population size stays constant throughout.

    Termination: after max_offspring total offspring generated.
    Chu & Beasley used 2,000,000 for large instances.
    We use 10,000 here for quick testing — increase for real experiments.
    """

    # precompute efficiency scores once
    efficiency_scores = compute_efficiency_scores(values, weights,
                                                   capacities, n, m)

    # initialize population
    population = initialize_population(population_size, n)

    # repair every initial solution and compute fitness
    for idx in range(population_size):
        population[idx] = repair(population[idx], values, weights,
                                  capacities, efficiency_scores, n, m)

    fitness_scores = [fitness(sol, values, n) for sol in population]

    # track best solution found
    best_fitness  = max(fitness_scores)
    best_solution = population[fitness_scores.index(best_fitness)].copy()

    # track feasibility rate at generation 0
    feasibility_rate_gen0 = sum(
        1 for sol in population
        if is_feasible(sol, weights, capacities, n, m)
    ) / population_size

    if verbose:
        print(f"Generation 0 | Best fitness: {best_fitness} | "
              f"Avg fitness: {sum(fitness_scores)/population_size:.1f} | "
              f"Feasibility rate: {feasibility_rate_gen0:.3f}")

    # main loop
    offspring_count = 0
    generation      = 0

    while offspring_count < max_offspring:

        # select two parents via binary tournament
        parent1 = tournament_select(population, fitness_scores)
        parent2 = tournament_select(population, fitness_scores)

        # crossover
        child = uniform_crossover(parent1, parent2, n)

        # mutation
        child = mutate(child, n)

        # repair — guarantees child is feasible
        child = repair(child, values, weights, capacities,
                        efficiency_scores, n, m)

        # evaluate child fitness
        child_fitness = fitness(child, values, n)

        # steady state replacement — replace worst if child is better
        worst_idx     = fitness_scores.index(min(fitness_scores))
        worst_fitness = fitness_scores[worst_idx]

        if child_fitness > worst_fitness:
            population[worst_idx]    = child
            fitness_scores[worst_idx] = child_fitness

        # update best
        if child_fitness > best_fitness:
            best_fitness  = child_fitness
            best_solution = child.copy()

        offspring_count += 1
        generation      += 1

        # print progress every 1000 offspring
        if verbose and offspring_count % 1000 == 0:
            print(f"Offspring {offspring_count:>6} | "
                  f"Best fitness: {best_fitness} | "
                  f"Avg fitness: {sum(fitness_scores)/population_size:.1f}")

    return best_solution, best_fitness

In [12]:
# ─────────────────────────────────────────────────────────────
# TEST — small hand-crafted instance
# ─────────────────────────────────────────────────────────────

# small test instance — 5 items, 2 constraints
# you can verify this by hand
n = 5
m = 2

values = [10, 6, 4, 8, 5]

# weights[j][i] = weight of item j under constraint i
weights = [
    [2, 3],   # item 1
    [4, 1],   # item 2
    [1, 2],   # item 3
    [3, 4],   # item 4
    [2, 2],   # item 5
]

capacities = [7, 8]

print("=" * 55)
print("Chu & Beasley GA — small test instance")
print(f"n={n} items, m={m} constraints")
print(f"Capacities: {capacities}")
print("=" * 55)

best_sol, best_fit = chu_beasley_ga(
    n, m, values, weights, capacities,
    population_size=20,   # small for testing
    max_offspring=500,    # small for testing
    verbose=True
)

print("=" * 55)
print(f"Best solution found: {best_sol}")
print(f"Best fitness:        {best_fit}")
print(f"Feasible:            "
        f"{is_feasible(best_sol, weights, capacities, n, m)}")

Chu & Beasley GA — small test instance
n=5 items, m=2 constraints
Capacities: [7, 8]
Generation 0 | Best fitness: 20 | Avg fitness: 17.6 | Feasibility rate: 1.000
Best solution found: [1, 1, 1, 0, 0]
Best fitness:        20
Feasible:            True


In [13]:
def parse_orlib_mkp(filepath):
    """
    Parse an OR-Library MKP instance file.

    Format:
        First line: number of instances in file
        For each instance:
            Line 1: n  m  optimal_value (or 0 if unknown)
            Line 2: values v_1 v_2 ... v_n
            Lines 3 to m+2: weights for each constraint
                            w_i1 w_i2 ... w_in
            Last line: capacities W_1 W_2 ... W_m
    """
    instances = []

    with open(filepath, 'r') as f:
        lines = [line.strip() for line in f.readlines()
                 if line.strip()]

    idx = 0
    num_instances = int(lines[idx]); idx += 1

    for _ in range(num_instances):

        # first line of instance: n m optimal
        parts = lines[idx].split(); idx += 1
        n       = int(parts[0])
        m       = int(parts[1])
        optimal = int(parts[2])

        # values
        values = []
        while len(values) < n:
            values += [int(x) for x in lines[idx].split()]
            idx += 1

        # weights — m rows each of length n
        # weights[j][i] = weight of item j under constraint i
        # OR-Library stores constraint i as a row of n values
        # so we read m rows and transpose
        constraint_rows = []
        for i in range(m):
            row = []
            while len(row) < n:
                row += [int(x) for x in lines[idx].split()]
                idx += 1
            constraint_rows.append(row)

        # transpose: weights[j][i]
        weights = [[constraint_rows[i][j] for i in range(m)]
                   for j in range(n)]

        # capacities
        capacities = []
        while len(capacities) < m:
            capacities += [int(x) for x in lines[idx].split()]
            idx += 1

        instances.append({
            'n':          n,
            'm':          m,
            'optimal':    optimal,
            'values':     values,
            'weights':    weights,
            'capacities': capacities
        })

    return instances

In [14]:
# Hard coding optimal results instead of writing another parser
best_known_mknapcb1 = [
    24381, 24274, 23551, 23534, 23991,   # 00-04  alpha=0.25
    24613, 25591, 23410, 24216, 24411,   # 05-09  alpha=0.25
    42757, 42545, 41968, 45090, 42218,   # 10-14  alpha=0.50
    42927, 42009, 45020, 43441, 44554,   # 15-19  alpha=0.50
    59822, 62081, 59802, 60479, 61091,   # 20-24  alpha=0.75
    58959, 61538, 61520, 59453, 59965    # 25-29  alpha=0.75
]

In [6]:
def run_orlib_test(filepath, best_known, max_offspring=50000,
                   population_size=200):

    instances = parse_orlib_mkp(filepath)

    print(f"{'Instance':<12} {'Best Known':<12} {'Found':<10} "
          f"{'Gap %':<10} {'Feasible'}")
    print("-" * 58)

    for idx, inst in enumerate(instances):

        best_sol, best_fit = chu_beasley_ga(
            inst['n'], inst['m'],
            inst['values'], inst['weights'], inst['capacities'],
            population_size=population_size,
            max_offspring=max_offspring,
            verbose=False
        )

        feasible = is_feasible(
            best_sol, inst['weights'],
            inst['capacities'], inst['n'], inst['m']
        )

        known   = best_known[idx]
        gap     = (known - best_fit) / known * 100

        print(f"Instance {idx+1:<3} {known:<12} {best_fit:<10} "
              f"{gap:<9.2f}% {feasible}")


# run on just the first 10 instances (alpha=0.25, hardest) first
run_orlib_test(
    'mknapcb1.txt',
    best_known=best_known_mknapcb1,
    max_offspring=10000,   # keep small for first test run
    population_size=50     # keep small for first test run
)

Instance     Best Known   Found      Gap %      Feasible
----------------------------------------------------------


KeyboardInterrupt: 

In [ ]:
import random

random.seed(42)
run_orlib_test('mknapcb1.txt', best_known_mknapcb1,
               max_offspring=10000, population_size=50)

print()

random.seed(99)
run_orlib_test('mknapcb1.txt', best_known_mknapcb1,
               max_offspring=10000, population_size=50)

Instance     Best Known   Found      Gap %      Feasible
----------------------------------------------------------
Instance 1   24381        24287      0.39     % True
Instance 2   24274        24230      0.18     % True
Instance 3   23551        23538      0.06     % True
Instance 4   23534        23323      0.90     % True
Instance 5   23991        23847      0.60     % True
Instance 6   24613        24495      0.48     % True
Instance 7   25591        25459      0.52     % True
Instance 8   23410        23358      0.22     % True
Instance 9   24216        24095      0.50     % True
Instance 10  24411        24269      0.58     % True
Instance 11  42757        42705      0.12     % True
Instance 12  42545        42388      0.37     % True
Instance 13  41968        41898      0.17     % True
Instance 14  45090        44926      0.36     % True
Instance 15  42218        42119      0.23     % True
Instance 16  42927        42698      0.53     % True
Instance 17  42009        41877     

In [15]:
# ─────────────────────────────────────────────────────────────
# PART 2A — EXACT GENERATING FUNCTION EXPANSION
# ─────────────────────────────────────────────────────────────
#
# What this does:
# For each item j we compute the sub-generating function g_j
# by multiplying together the factors for all OTHER items.
# Each factor is (1 + x^w) for single constraint
# or (1 + x1^w1 * x2^w2 * ... * xm^wm) for multiple constraints.
#
# We represent the polynomial as a dictionary where:
#   key   = tuple of exponents (k1, k2, ..., km)
#   value = coefficient (how many subsets have that weight combination)
#
# Example for m=1:
#   {(0,): 1, (2,): 1, (3,): 1, (5,): 1}
#   means: 1 subset of weight 0, 1 of weight 2, 1 of weight 3, 1 of weight 5
#
# Example for m=2:
#   {(0,0): 1, (2,3): 1, (4,2): 1, (6,5): 1}
#   means: 1 subset with (c1_weight=0, c2_weight=0), etc.

def multiply_polynomials(poly, factor_weight):
    """
    Multiply an existing polynomial by one new factor (1 + x^factor_weight).

    factor_weight is a tuple of m weights, one per constraint.
    e.g. for m=1: (3,)
         for m=2: (3, 5)

    Multiplying by (1 + x^w) means:
        new_poly = poly * 1  +  poly * x^w

        The first part keeps all existing terms unchanged.
        The second part shifts every existing term's exponents
        by adding factor_weight to each dimension.
    """
    new_poly = {}

    for exponent_tuple, coefficient in poly.items():

        # term * 1 — keep the existing term unchanged
        if exponent_tuple in new_poly:
            new_poly[exponent_tuple] += coefficient
        else:
            new_poly[exponent_tuple] = coefficient

        # term * x^factor_weight — shift exponents by adding factor_weight
        shifted = tuple(exponent_tuple[i] + factor_weight[i]
                        for i in range(len(factor_weight)))

        if shifted in new_poly:
            new_poly[shifted] += coefficient
        else:
            new_poly[shifted] = coefficient

    return new_poly


def compute_sub_generating_function(weights, j, n, m):
    """
    Compute g_j — the sub-generating function for item j.
    This is the product of factors for all items EXCEPT item j.

    Start with polynomial = {(0,0,...,0): 1}
    which represents the empty subset with weight 0 in all constraints.

    Then multiply by each item k's factor (1 + x^w_k) for k != j.
    """
    # start with the empty subset — weight 0 in all constraints
    zero_exponent = tuple(0 for _ in range(m))
    poly = {zero_exponent: 1}

    for k in range(n):
        if k == j:
            continue  # skip item j — this is g_j not f

        # factor weight for item k: tuple of weights across all constraints
        factor_weight = tuple(weights[k][i] for i in range(m))
        poly = multiply_polynomials(poly, factor_weight)

    return poly


def compute_feasibility_densities_exact(weights, capacities, n, m):
    """
    Compute exact feasibility densities rho_j^1 and rho_j^0
    for every item j using generating function expansion.

    rho_j^1 = fraction of terms in g_j where exponent[i] <= W_i - w_ji for all i
              (feasible completions when item j is INCLUDED)

    rho_j^0 = fraction of terms in g_j where exponent[i] <= W_i for all i
              (feasible completions when item j is EXCLUDED)

    Total number of terms in g_j = 2^(n-1)
    (each of the other n-1 items is either included or excluded)
    """
    total_terms = 2 ** (n - 1)
    results = []

    for j in range(n):

        # compute sub-generating function for item j
        g_j = compute_sub_generating_function(weights, j, n, m)

        count_in  = 0  # feasible when item j included
        count_out = 0  # feasible when item j excluded

        for exponent_tuple, coefficient in g_j.items():

            # check feasibility when item j is INCLUDED
            # remaining capacity = W_i - w_ji for each constraint i
            feasible_in = all(
                exponent_tuple[i] <= capacities[i] - weights[j][i]
                for i in range(m)
            )

            # check feasibility when item j is EXCLUDED
            # full capacity W_i available
            feasible_out = all(
                exponent_tuple[i] <= capacities[i]
                for i in range(m)
            )

            # coefficient tells us how many subsets have this weight combination
            # so we add the coefficient not just 1
            if feasible_in:
                count_in  += coefficient
            if feasible_out:
                count_out += coefficient

        rho_in  = count_in  / total_terms
        rho_out = count_out / total_terms

        if rho_in + rho_out == 0:
            p_j = 0.0
        else:
            p_j = rho_in / (rho_in + rho_out)

        results.append({
            'item':     j + 1,
            'rho_in':   round(rho_in,  6),
            'rho_out':  round(rho_out, 6),
            'p_j':      round(p_j,     6)
        })

    return results


In [16]:
print("\n" + "=" * 60)
print("LARGER SINGLE CONSTRAINT: 6 items, capacity=15")
print("=" * 60)

weights_3    = [[3], [5], [2], [7], [4], [6]]
capacities_3 = [15]
n3, m3       = 6, 1

results_3 = compute_feasibility_densities_exact(weights_3, capacities_3,
                                                 n3, m3)
print(f"\n{'Item':<8} {'weight':<10} {'rho_in':<12} {'rho_out':<12} {'p_j':<10}")
print("-" * 55)
for i, r in enumerate(results_3):
    print(f"{r['item']:<8} {weights_3[i][0]:<10} "
          f"{r['rho_in']:<12} {r['rho_out']:<12} {r['p_j']:<10}")


print("\n" + "=" * 60)
print("LARGER TWO CONSTRAINT: 8 items, W1=20, W2=18")
print("=" * 60)

weights_4 = [
    [3, 2], [5, 4], [2, 6], [7, 3],
    [4, 5], [6, 1], [1, 7], [8, 4]
]
capacities_4 = [20, 18]
n4, m4       = 8, 2

results_4 = compute_feasibility_densities_exact(weights_4, capacities_4,
                                                 n4, m4)
print(f"\n{'Item':<8} {'w(C1)':<8} {'w(C2)':<8} "
      f"{'rho_in':<12} {'rho_out':<12} {'p_j':<10}")
print("-" * 60)
for i, r in enumerate(results_4):
    print(f"{r['item']:<8} {weights_4[i][0]:<8} {weights_4[i][1]:<8} "
          f"{r['rho_in']:<12} {r['rho_out']:<12} {r['p_j']:<10}")


LARGER SINGLE CONSTRAINT: 6 items, capacity=15

Item     weight     rho_in       rho_out      p_j       
-------------------------------------------------------
1        3          0.53125      0.71875      0.425     
2        5          0.46875      0.78125      0.375     
3        2          0.5625       0.6875       0.45      
4        7          0.375        0.875        0.3       
5        4          0.5          0.75         0.4       
6        6          0.4375       0.8125       0.35      

LARGER TWO CONSTRAINT: 8 items, W1=20, W2=18

Item     w(C1)    w(C2)    rho_in       rho_out      p_j       
------------------------------------------------------------
1        3        2        0.4375       0.59375      0.424242  
2        5        4        0.375        0.65625      0.363636  
3        2        6        0.390625     0.640625     0.378788  
4        7        3        0.351562     0.679688     0.340909  
5        4        5        0.382812     0.648438     0.371212  
6   

In [17]:


# ─────────────────────────────────────────────────────────────
# VERIFY AGAINST YOUR HAND CALCULATIONS FROM THE LATEX DOCUMENT
# ─────────────────────────────────────────────────────────────

print("=" * 60)
print("SINGLE CONSTRAINT EXAMPLE")
print("Items: weights=(3,2,4), capacity=5")
print("Expected from LaTeX document:")
print("  Item 1: rho_in=0.50, rho_out=0.75, p_j=0.40")
print("  Item 2: rho_in=0.50, rho_out=0.75, p_j=0.40")
print("  Item 3: rho_in=0.25, rho_out=1.00, p_j=0.20")
print("=" * 60)

weights_1    = [[3], [2], [4]]
capacities_1 = [5]
n1, m1       = 3, 1

results_1 = compute_feasibility_densities_exact(weights_1, capacities_1,
                                                 n1, m1)

print(f"\n{'Item':<8} {'rho_in':<12} {'rho_out':<12} {'p_j':<10}")
print("-" * 45)
for r in results_1:
    print(f"{r['item']:<8} {r['rho_in']:<12} {r['rho_out']:<12} {r['p_j']:<10}")


print()
print("=" * 60)
print("TWO CONSTRAINT EXAMPLE")
print("Capacities: W1=7, W2=8")
print("Expected from LaTeX document:")
print("  Item 1: rho_in=0.500, rho_out=0.625, p_j=0.444")
print("  Item 2: rho_in=0.375, rho_out=0.750, p_j=0.333")
print("  Item 3: rho_in=0.375, rho_out=0.750, p_j=0.333")
print("  Item 4: rho_in=0.250, rho_out=0.875, p_j=0.222")
print("=" * 60)

weights_2    = [[2,3], [4,2], [3,5], [5,1]]
capacities_2 = [7, 8]
n2, m2       = 4, 2

results_2 = compute_feasibility_densities_exact(weights_2, capacities_2,
                                                 n2, m2)

print(f"\n{'Item':<8} {'rho_in':<12} {'rho_out':<12} {'p_j':<10}")
print("-" * 45)
for r in results_2:
    print(f"{r['item']:<8} {r['rho_in']:<12} {r['rho_out']:<12} {r['p_j']:<10}")

SINGLE CONSTRAINT EXAMPLE
Items: weights=(3,2,4), capacity=5
Expected from LaTeX document:
  Item 1: rho_in=0.50, rho_out=0.75, p_j=0.40
  Item 2: rho_in=0.50, rho_out=0.75, p_j=0.40
  Item 3: rho_in=0.25, rho_out=1.00, p_j=0.20

Item     rho_in       rho_out      p_j       
---------------------------------------------
1        0.5          0.75         0.4       
2        0.5          0.75         0.4       
3        0.25         1.0          0.2       

TWO CONSTRAINT EXAMPLE
Capacities: W1=7, W2=8
Expected from LaTeX document:
  Item 1: rho_in=0.500, rho_out=0.625, p_j=0.444
  Item 2: rho_in=0.375, rho_out=0.750, p_j=0.333
  Item 3: rho_in=0.375, rho_out=0.750, p_j=0.333
  Item 4: rho_in=0.250, rho_out=0.875, p_j=0.222

Item     rho_in       rho_out      p_j       
---------------------------------------------
1        0.5          0.625        0.444444  
2        0.375        0.75         0.333333  
3        0.375        0.75         0.333333  
4        0.25         0.875        0

In [18]:
def compute_feasibility_densities(weights, capacities, num_items, num_samples=5000):
    num_constraints = len(capacities)
    probabilities = []
    for j in range(num_items):
        count_in  = 0
        count_out = 0
        for _ in range(num_samples):
            subset_weights = [0] * num_constraints
            for k in range(num_items):
                if k == j:
                    continue
                if random.random() < 0.5:
                    for i in range(num_constraints):
                        subset_weights[i] += weights[k][i]
            feasible_in = all(
                subset_weights[i] <= capacities[i] - weights[j][i]
                for i in range(num_constraints)
            )
            feasible_out = all(
                subset_weights[i] <= capacities[i]
                for i in range(num_constraints)
            )
            if feasible_in:  count_in  += 1
            if feasible_out: count_out += 1
        rho_in  = count_in  / num_samples
        rho_out = count_out / num_samples
        p_j = rho_in / (rho_in + rho_out) if (rho_in + rho_out) > 0 else 0.0
        probabilities.append({'item': j+1, 'rho_in': rho_in,
                               'rho_out': rho_out, 'p_j': p_j})
    return probabilities

In [19]:
def run_ga(n, m, values, weights, capacities,
           population_size=50,
           max_offspring=10000,
           best_known=None,
           target_pct=0.99,
           init_method='uniform',
           p_j_values=None,
           seed=None):

    if seed is not None:
        random.seed(seed)

    efficiency_scores = compute_efficiency_scores(
        values, weights, capacities, n, m
    )

    if init_method == 'uniform':
        population = initialize_population(population_size, n)
    else:
        population = initialize_population_biased(
            population_size, n, p_j_values
        )

    # measure feasibility BEFORE repair
    feasible_before_repair = sum(
        1 for sol in population
        if is_feasible(sol, weights, capacities, n, m)
    ) / population_size

    # repair every initial solution
    for idx in range(population_size):
        population[idx] = repair(
            population[idx], values, weights,
            capacities, efficiency_scores, n, m
        )

    fitness_scores    = [fitness(sol, values, n) for sol in population]
    avg_fitness_gen0  = sum(fitness_scores) / population_size
    best_fitness      = max(fitness_scores)
    best_solution     = population[fitness_scores.index(best_fitness)].copy()

    convergence_point = None
    if best_known is not None:
        if best_fitness >= best_known * target_pct:
            convergence_point = 0

    offspring_count = 0

    while offspring_count < max_offspring:

        parent1 = tournament_select(population, fitness_scores)
        parent2 = tournament_select(population, fitness_scores)
        child   = uniform_crossover(parent1, parent2, n)
        child   = mutate(child, n)
        child   = repair(child, values, weights,
                         capacities, efficiency_scores, n, m)

        child_fitness = fitness(child, values, n)
        worst_idx     = fitness_scores.index(min(fitness_scores))

        if child_fitness > fitness_scores[worst_idx]:
            population[worst_idx]     = child
            fitness_scores[worst_idx] = child_fitness

        if child_fitness > best_fitness:
            best_fitness  = child_fitness
            best_solution = child.copy()

        offspring_count += 1

        if (convergence_point is None and
                best_known is not None and
                best_fitness >= best_known * target_pct):
            convergence_point = offspring_count

    return {
        'feasibility_rate_gen0': feasible_before_repair,
        'avg_fitness_gen0':      avg_fitness_gen0,
        'best_fitness':          best_fitness,
        'convergence_point':     convergence_point,
        'gap_pct': ((best_known - best_fitness) / best_known * 100
                    if best_known else None)
    }

def initialize_population_biased(population_size, n, p_j_values):
    population = []
    for _ in range(population_size):
        solution = []
        for j in range(n):
            if random.random() < p_j_values[j]:
                solution.append(1)
            else:
                solution.append(0)
        population.append(solution)
    return population

In [20]:
import random

def generate_random_mkp_instance(n, m, tightness=0.25, seed=42):
    random.seed(seed)
    values  = [random.randint(1, 100) for _ in range(n)]
    weights = [[random.randint(1, 100) for _ in range(m)]
               for _ in range(n)]
    capacities = []
    for i in range(m):
        total_weight_i = sum(weights[j][i] for j in range(n))
        capacities.append(int(tightness * total_weight_i))
    return values, weights, capacities


def quick_comparison_test(n=100, m=3, tightness=0.25,
                           num_runs=5,
                           population_size=50,
                           max_offspring=5000,
                           num_samples=5000):

    print("=" * 65)
    print(f"MKP INSTANCE: n={n} items, m={m} constraints, "
          f"tightness α={tightness}")
    print(f"Runs per method: {num_runs} | "
          f"Population: {population_size} | "
          f"Max offspring: {max_offspring}")
    print("=" * 65)

    values, weights, capacities = generate_random_mkp_instance(
        n, m, tightness, seed=42
    )

    print(f"\nCapacities: {capacities}")
    print(f"Value range: [{min(values)}, {max(values)}]")

    for i in range(m):
        total = sum(weights[j][i] for j in range(n))
        ratio = capacities[i] / total
        print(f"Constraint {i+1}: capacity={capacities[i]}, "
              f"total_weight={total}, ratio={ratio:.3f}")

    # ── compute p_j via Monte Carlo ─────────────────────────
    print(f"\nComputing p_j values via Monte Carlo "
          f"({num_samples} samples per item)...")

    p_j_data   = compute_feasibility_densities(
        weights, capacities, n, num_samples=num_samples
    )
    p_j_values = [r['p_j'] for r in p_j_data]

    print(f"p_j range: [{min(p_j_values):.3f}, {max(p_j_values):.3f}]")
    print(f"p_j mean:  {sum(p_j_values)/len(p_j_values):.3f} "
          f"(Chu & Beasley always uses 0.500)")

    # ── find reference best ─────────────────────────────────
    print("\nFinding reference best solution (one long uniform run)...")
    ref_result = run_ga(
        n, m, values, weights, capacities,
        population_size=100,
        max_offspring=50000,
        best_known=None,
        init_method='uniform',
        seed=999
    )
    reference_best = ref_result['best_fitness']
    print(f"Reference best: {reference_best}")

    # ── run both methods ────────────────────────────────────
    uniform_feas = []
    biased_feas  = []
    uniform_conv = []
    biased_conv  = []
    uniform_gaps = []
    biased_gaps  = []

    print(f"\nRunning {num_runs} trials per method...\n")

    for run in range(num_runs):

        u = run_ga(
            n, m, values, weights, capacities,
            population_size=population_size,
            max_offspring=max_offspring,
            best_known=reference_best,
            target_pct=0.99,
            init_method='uniform',
            seed=run * 10
        )
        uniform_feas.append(u['feasibility_rate_gen0'])
        uniform_conv.append(u['convergence_point'])
        uniform_gaps.append(u['gap_pct'])

        b = run_ga(
            n, m, values, weights, capacities,
            population_size=population_size,
            max_offspring=max_offspring,
            best_known=reference_best,
            target_pct=0.99,
            init_method='biased',
            p_j_values=p_j_values,
            seed=run * 10
        )
        biased_feas.append(b['feasibility_rate_gen0'])
        biased_conv.append(b['convergence_point'])
        biased_gaps.append(b['gap_pct'])

        print(f"  Run {run+1}: "
              f"Uniform feas={u['feasibility_rate_gen0']:.3f} "
              f"conv={u['convergence_point']}  |  "
              f"Biased  feas={b['feasibility_rate_gen0']:.3f} "
              f"conv={b['convergence_point']}")

    # ── summary ─────────────────────────────────────────────
    def avg(lst):
        vals = [x for x in lst if x is not None]
        return sum(vals) / len(vals) if vals else None

    def pct_never(lst):
        return sum(1 for x in lst if x is None) / len(lst) * 100

    u_avg_feas = avg(uniform_feas)
    b_avg_feas = avg(biased_feas)
    u_avg_conv = avg(uniform_conv)
    b_avg_conv = avg(biased_conv)
    u_avg_gap  = avg(uniform_gaps)
    b_avg_gap  = avg(biased_gaps)

    print("\n" + "=" * 65)
    print("RESULTS SUMMARY")
    print("=" * 65)
    print(f"\n{'Metric':<35} {'Uniform':>12} {'Biased':>12}")
    print("-" * 62)

    print(f"{'Feasibility rate at gen 0':<35} "
          f"{u_avg_feas:>12.3f} "
          f"{b_avg_feas:>12.3f}")

    if u_avg_feas and b_avg_feas:
        improvement_feas = (b_avg_feas - u_avg_feas) * 100
        print(f"{'  improvement':<35} "
              f"{'':>12} "
              f"{improvement_feas:>+11.1f}pp")

    conv_u_str = f"{u_avg_conv:.0f}" if u_avg_conv else "never"
    conv_b_str = f"{b_avg_conv:.0f}" if b_avg_conv else "never"
    print(f"\n{'Offspring to reach 99% of best':<35} "
          f"{conv_u_str:>12} "
          f"{conv_b_str:>12}")

    if u_avg_conv and b_avg_conv:
        improvement_conv = (u_avg_conv - b_avg_conv) / u_avg_conv * 100
        print(f"{'  improvement':<35} "
              f"{'':>12} "
              f"{improvement_conv:>+11.1f}%")

    print(f"\n{'Never reached 99% (uniform)':<35} "
          f"{pct_never(uniform_conv):>11.0f}%")
    print(f"{'Never reached 99% (biased)':<35} "
          f"{pct_never(biased_conv):>11.0f}%")

    gap_u_str = f"{u_avg_gap:.3f}" if u_avg_gap is not None else "N/A"
    gap_b_str = f"{b_avg_gap:.3f}" if b_avg_gap is not None else "N/A"
    print(f"\n{'Final gap from reference (%)':<35} "
          f"{gap_u_str:>12} "
          f"{gap_b_str:>12}")

    print("\n" + "=" * 65)

    return {
        'uniform_feas': u_avg_feas,
        'biased_feas':  b_avg_feas,
        'uniform_conv': u_avg_conv,
        'biased_conv':  b_avg_conv,
    }


# ── run it ──────────────────────────────────────────────────
quick_comparison_test(
    n=100,
    m=3,
    tightness=0.25,
    num_runs=5,
    population_size=50,
    max_offspring=5000
)

MKP INSTANCE: n=100 items, m=3 constraints, tightness α=0.25
Runs per method: 5 | Population: 50 | Max offspring: 5000

Capacities: [1280, 1279, 1194]
Value range: [1, 99]
Constraint 1: capacity=1280, total_weight=5123, ratio=0.250
Constraint 2: capacity=1279, total_weight=5118, ratio=0.250
Constraint 3: capacity=1194, total_weight=4776, ratio=0.250

Computing p_j values via Monte Carlo (5000 samples per item)...


KeyboardInterrupt: 

In [21]:
import torch
import time

def compute_pj_gpu(weights, capacities, n, m, num_samples=10_000_000):

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    print(f"Samples per item: {num_samples:,}")
    print(f"Total samples: {num_samples * n:,}\n")

    W = torch.tensor(weights, dtype=torch.float32, device=device)
    C = torch.tensor(capacities, dtype=torch.float32, device=device)

    results = []
    batch_size = 500_000
    num_batches = num_samples // batch_size

    start = time.time()

    for j in range(n):

        other_idx = [k for k in range(n) if k != j]
        W_other   = W[other_idx, :]       # (n-1, m)
        rem_in    = C - W[j]              # (m,)

        count_in  = 0
        count_out = 0

        for _ in range(num_batches):
            samples = torch.randint(0, 2,
                                    (batch_size, n - 1),
                                    dtype=torch.float32,
                                    device=device)
            totals  = samples @ W_other   # (batch_size, m)

            count_in  += torch.all(totals <= rem_in, dim=1).sum().item()
            count_out += torch.all(totals <= C,      dim=1).sum().item()

        actual = num_batches * batch_size
        rho_in  = count_in  / actual
        rho_out = count_out / actual
        denom   = rho_in + rho_out
        p_j     = max(0.05, rho_in / denom if denom > 0 else 0.05)

        results.append({
            'item':    j + 1,
            'rho_in':  rho_in,
            'rho_out': rho_out,
            'p_j':     p_j
        })

        if (j + 1) % 10 == 0:
            elapsed = time.time() - start
            print(f"Item {j+1:>3}/{n} | "
                  f"rho_in={rho_in:.8f} | "
                  f"rho_out={rho_out:.8f} | "
                  f"p_j={p_j:.4f} | "
                  f"elapsed={elapsed:.1f}s")

    total_time = time.time() - start
    p_vals  = [r['p_j']    for r in results]
    nonzero = sum(1 for r in results if r['rho_in'] > 0)

    print(f"\n{'='*60}")
    print(f"DONE in {total_time:.1f}s")
    print(f"Items with nonzero rho_in: {nonzero}/{n}")
    print(f"p_j range: [{min(p_vals):.4f}, {max(p_vals):.4f}]")
    print(f"p_j mean:  {sum(p_vals)/len(p_vals):.4f}")
    print(f"{'='*60}")

    return results


# ── run it ───────────────────────────────────────────────────
values, weights, capacities = generate_random_mkp_instance(
    n=100, m=3, tightness=0.25, seed=42
)

print("="*60)
print("TIGHT INSTANCE: n=100, m=3, tightness=0.25")
print(f"Capacities: {capacities}")
print("="*60 + "\n")

p_j_data   = compute_pj_gpu(weights, capacities,
                              n=100, m=3,
                              num_samples=10_000_000)
p_j_values = [r['p_j'] for r in p_j_data]

TIGHT INSTANCE: n=100, m=3, tightness=0.25
Capacities: [1280, 1279, 1194]

Device: cuda
Samples per item: 10,000,000
Total samples: 1,000,000,000

Item  10/100 | rho_in=0.00000000 | rho_out=0.00000010 | p_j=0.0500 | elapsed=0.7s
Item  20/100 | rho_in=0.00000010 | rho_out=0.00000030 | p_j=0.2500 | elapsed=1.3s
Item  30/100 | rho_in=0.00000030 | rho_out=0.00000030 | p_j=0.5000 | elapsed=1.9s
Item  40/100 | rho_in=0.00000000 | rho_out=0.00000020 | p_j=0.0500 | elapsed=2.6s
Item  50/100 | rho_in=0.00000010 | rho_out=0.00000010 | p_j=0.5000 | elapsed=3.2s
Item  60/100 | rho_in=0.00000000 | rho_out=0.00000020 | p_j=0.0500 | elapsed=3.9s
Item  70/100 | rho_in=0.00000000 | rho_out=0.00000020 | p_j=0.0500 | elapsed=4.5s
Item  80/100 | rho_in=0.00000000 | rho_out=0.00000000 | p_j=0.0500 | elapsed=5.1s
Item  90/100 | rho_in=0.00000000 | rho_out=0.00000000 | p_j=0.0500 | elapsed=5.8s
Item 100/100 | rho_in=0.00000000 | rho_out=0.00000010 | p_j=0.0500 | elapsed=6.4s

DONE in 6.4s
Items with nonzero 

In [22]:
# p_j_values already computed from GPU run above
p_j_values = [r['p_j'] for r in p_j_data]

print("p_j values summary:")
print(f"Min: {min(p_j_values):.4f}")
print(f"Max: {max(p_j_values):.4f}")
print(f"Mean: {sum(p_j_values)/len(p_j_values):.4f}")
print(f"Items above 0.05 floor: {sum(1 for p in p_j_values if p > 0.05)}/100")
print()

# generate the same instance again with same seed
values, weights, capacities = generate_random_mkp_instance(
    n=100, m=3, tightness=0.25, seed=42
)

# find reference best
print("Finding reference best...")
ref_result = run_ga(
    100, 3, values, weights, capacities,
    population_size=100,
    max_offspring=50000,
    best_known=None,
    init_method='uniform',
    seed=999
)
reference_best = ref_result['best_fitness']
print(f"Reference best: {reference_best}")

# run comparison
print("\nRunning 5 trials per method...\n")

uniform_feas = []
biased_feas  = []
uniform_conv = []
biased_conv  = []

for run in range(5):

    u = run_ga(
        100, 3, values, weights, capacities,
        population_size=50,
        max_offspring=5000,
        best_known=reference_best,
        target_pct=0.99,
        init_method='uniform',
        seed=run * 10
    )

    b = run_ga(
        100, 3, values, weights, capacities,
        population_size=50,
        max_offspring=5000,
        best_known=reference_best,
        target_pct=0.99,
        init_method='biased',
        p_j_values=p_j_values,
        seed=run * 10
    )

    uniform_feas.append(u['feasibility_rate_gen0'])
    biased_feas.append(b['feasibility_rate_gen0'])
    uniform_conv.append(u['convergence_point'])
    biased_conv.append(b['convergence_point'])

    print(f"Run {run+1}: "
          f"Uniform feas={u['feasibility_rate_gen0']:.3f} "
          f"conv={u['convergence_point']} | "
          f"Biased feas={b['feasibility_rate_gen0']:.3f} "
          f"conv={b['convergence_point']}")

def avg(lst):
    vals = [x for x in lst if x is not None]
    return sum(vals)/len(vals) if vals else None

print(f"\n{'='*55}")
print(f"SUMMARY")
print(f"{'='*55}")
print(f"{'Metric':<35} {'Uniform':>10} {'Biased':>10}")
print(f"{'-'*55}")
print(f"{'Feasibility rate at gen 0':<35} "
      f"{avg(uniform_feas):>10.3f} "
      f"{avg(biased_feas):>10.3f}")
print(f"{'Offspring to 99% of best':<35} "
      f"{avg(uniform_conv) or 'never':>10} "
      f"{avg(biased_conv) or 'never':>10}")
if avg(uniform_conv) and avg(biased_conv):
    improvement = (avg(uniform_conv) - avg(biased_conv)) / avg(uniform_conv) * 100
    print(f"{'Convergence improvement':<35} {'':>10} {improvement:>+9.1f}%")


p_j values summary:
Min: 0.0500
Max: 0.5000
Mean: 0.1579
Items above 0.05 floor: 32/100

Finding reference best...
Reference best: 2358

Running 5 trials per method...

Run 1: Uniform feas=0.000 conv=30 | Biased feas=1.000 conv=63
Run 2: Uniform feas=0.000 conv=32 | Biased feas=1.000 conv=52
Run 3: Uniform feas=0.000 conv=63 | Biased feas=0.980 conv=67
Run 4: Uniform feas=0.000 conv=126 | Biased feas=0.980 conv=86
Run 5: Uniform feas=0.000 conv=40 | Biased feas=0.940 conv=107

SUMMARY
Metric                                 Uniform     Biased
-------------------------------------------------------
Feasibility rate at gen 0                0.000      0.980
Offspring to 99% of best                  58.2       75.0
Convergence improvement                            -28.9%


In [23]:
def run_ga_detailed(n, m, values, weights, capacities,
                    population_size=50,
                    max_offspring=5000,
                    best_known=None,
                    init_method='uniform',
                    p_j_values=None,
                    seed=None,
                    track_every=50):
    """
    Same GA but tracks fitness at regular intervals
    so we can see the convergence curve.
    """
    if seed is not None:
        random.seed(seed)

    efficiency_scores = compute_efficiency_scores(
        values, weights, capacities, n, m
    )

    if init_method == 'uniform':
        population = initialize_population(population_size, n)
    else:
        population = initialize_population_biased(
            population_size, n, p_j_values
        )

    # measure before repair
    feasible_before = sum(
        1 for sol in population
        if is_feasible(sol, weights, capacities, n, m)
    ) / population_size

    # repair
    for idx in range(population_size):
        population[idx] = repair(
            population[idx], values, weights,
            capacities, efficiency_scores, n, m
        )

    fitness_scores   = [fitness(sol, values, n) for sol in population]
    best_fitness     = max(fitness_scores)
    avg_fitness_gen0 = sum(fitness_scores) / population_size
    min_fitness_gen0 = min(fitness_scores)

    # what percentage of best known are we at gen 0
    pct_of_best_gen0 = (best_fitness / best_known * 100
                        if best_known else None)

    # convergence curve — track best fitness every track_every offspring
    curve = [(0, best_fitness, avg_fitness_gen0)]

    offspring_count  = 0
    convergence_99   = None
    convergence_95   = None
    convergence_90   = None

    while offspring_count < max_offspring:

        parent1 = tournament_select(population, fitness_scores)
        parent2 = tournament_select(population, fitness_scores)
        child   = uniform_crossover(parent1, parent2, n)
        child   = mutate(child, n)
        child   = repair(child, values, weights,
                         capacities, efficiency_scores, n, m)

        child_fitness = fitness(child, values, n)
        worst_idx     = fitness_scores.index(min(fitness_scores))

        if child_fitness > fitness_scores[worst_idx]:
            population[worst_idx]     = child
            fitness_scores[worst_idx] = child_fitness

        if child_fitness > best_fitness:
            best_fitness = child_fitness

        offspring_count += 1

        # track convergence thresholds
        if best_known:
            if convergence_99 is None and best_fitness >= best_known * 0.99:
                convergence_99 = offspring_count
            if convergence_95 is None and best_fitness >= best_known * 0.95:
                convergence_95 = offspring_count
            if convergence_90 is None and best_fitness >= best_known * 0.90:
                convergence_90 = offspring_count

        # record curve point
        if offspring_count % track_every == 0:
            avg_fit = sum(fitness_scores) / population_size
            curve.append((offspring_count, best_fitness, avg_fit))

    return {
        'feasibility_rate_gen0': feasible_before,
        'best_fitness_gen0':     best_fitness if offspring_count == 0
                                 else curve[0][1],
        'avg_fitness_gen0':      avg_fitness_gen0,
        'min_fitness_gen0':      min_fitness_gen0,
        'pct_of_best_gen0':      pct_of_best_gen0,
        'best_fitness_final':    best_fitness,
        'pct_of_best_final':     best_fitness / best_known * 100
                                 if best_known else None,
        'convergence_90':        convergence_90,
        'convergence_95':        convergence_95,
        'convergence_99':        convergence_99,
        'curve':                 curve
    }


def detailed_comparison(values, weights, capacities,
                         p_j_values, reference_best,
                         n=100, m=3,
                         num_runs=5,
                         population_size=50,
                         max_offspring=5000,
                         track_every=50):

    print(f"Reference best: {reference_best}")
    print(f"90% threshold:  {reference_best * 0.90:.0f}")
    print(f"95% threshold:  {reference_best * 0.95:.0f}")
    print(f"99% threshold:  {reference_best * 0.99:.0f}")
    print()

    u_results = []
    b_results = []

    for run in range(num_runs):

        u = run_ga_detailed(
            n, m, values, weights, capacities,
            population_size=population_size,
            max_offspring=max_offspring,
            best_known=reference_best,
            init_method='uniform',
            seed=run * 10,
            track_every=track_every
        )
        u_results.append(u)

        b = run_ga_detailed(
            n, m, values, weights, capacities,
            population_size=population_size,
            max_offspring=max_offspring,
            best_known=reference_best,
            init_method='biased',
            p_j_values=p_j_values,
            seed=run * 10,
            track_every=track_every
        )
        b_results.append(b)

        print(f"Run {run+1}:")
        print(f"  Uniform | feas={u['feasibility_rate_gen0']:.3f} | "
              f"gen0_best={u['avg_fitness_gen0']:.0f} "
              f"({u['pct_of_best_gen0']:.1f}% of best) | "
              f"conv90={u['convergence_90']} "
              f"conv95={u['convergence_95']} "
              f"conv99={u['convergence_99']}")
        print(f"  Biased  | feas={b['feasibility_rate_gen0']:.3f} | "
              f"gen0_best={b['avg_fitness_gen0']:.0f} "
              f"({b['pct_of_best_gen0']:.1f}% of best) | "
              f"conv90={b['convergence_90']} "
              f"conv95={b['convergence_95']} "
              f"conv99={b['convergence_99']}")
        print()

    def avg(lst, key):
        vals = [x[key] for x in lst if x[key] is not None]
        return sum(vals) / len(vals) if vals else None

    print("=" * 65)
    print("SUMMARY ACROSS ALL RUNS")
    print("=" * 65)
    print(f"\n{'Metric':<40} {'Uniform':>10} {'Biased':>10}")
    print("-" * 65)
    print(f"{'Feasibility rate at gen 0':<40} "
          f"{avg(u_results,'feasibility_rate_gen0'):>10.3f} "
          f"{avg(b_results,'feasibility_rate_gen0'):>10.3f}")
    print(f"{'Avg fitness at gen 0 (abs)':<40} "
          f"{avg(u_results,'avg_fitness_gen0'):>10.1f} "
          f"{avg(b_results,'avg_fitness_gen0'):>10.1f}")
    print(f"{'Avg fitness at gen 0 (% of best)':<40} "
          f"{avg(u_results,'pct_of_best_gen0'):>9.1f}% "
          f"{avg(b_results,'pct_of_best_gen0'):>9.1f}%")
    print(f"{'Final best fitness (% of best)':<40} "
          f"{avg(u_results,'pct_of_best_final'):>9.1f}% "
          f"{avg(b_results,'pct_of_best_final'):>9.1f}%")
    print()
    print(f"{'Offspring to reach 90% of best':<40} "
          f"{avg(u_results,'convergence_90') or 'never':>10} "
          f"{avg(b_results,'convergence_90') or 'never':>10}")
    print(f"{'Offspring to reach 95% of best':<40} "
          f"{avg(u_results,'convergence_95') or 'never':>10} "
          f"{avg(b_results,'convergence_95') or 'never':>10}")
    print(f"{'Offspring to reach 99% of best':<40} "
          f"{avg(u_results,'convergence_99') or 'never':>10} "
          f"{avg(b_results,'convergence_99') or 'never':>10}")

    # print convergence curve for run 1 of each method
    print("\n" + "=" * 65)
    print("CONVERGENCE CURVE (Run 1) — best fitness every 50 offspring")
    print("=" * 65)
    print(f"\n{'Offspring':<12} {'Uniform best':>14} {'% of best':>10} "
          f"{'Biased best':>14} {'% of best':>10}")
    print("-" * 65)

    u_curve = u_results[0]['curve']
    b_curve = b_results[0]['curve']

    for i in range(min(len(u_curve), len(b_curve))):
        u_pt = u_curve[i]
        b_pt = b_curve[i]
        print(f"{u_pt[0]:<12} "
              f"{u_pt[1]:>14.0f} "
              f"{u_pt[1]/reference_best*100:>9.1f}% "
              f"{b_pt[1]:>14.0f} "
              f"{b_pt[1]/reference_best*100:>9.1f}%")

    return u_results, b_results


# ── run it ───────────────────────────────────────────────────
u_results, b_results = detailed_comparison(
    values, weights, capacities,
    p_j_values=p_j_values,
    reference_best=reference_best,
    n=100, m=3,
    num_runs=5,
    population_size=50,
    max_offspring=5000,
    track_every=50
)

Reference best: 2358
90% threshold:  2122
95% threshold:  2240
99% threshold:  2334

Run 1:
  Uniform | feas=0.000 | gen0_best=1935 (95.0% of best) | conv90=1 conv95=6 conv99=30
  Biased  | feas=1.000 | gen0_best=1749 (91.4% of best) | conv90=1 conv95=2 conv99=63

Run 2:
  Uniform | feas=0.000 | gen0_best=1934 (94.6% of best) | conv90=1 conv95=18 conv99=32
  Biased  | feas=1.000 | gen0_best=1809 (93.5% of best) | conv90=1 conv95=31 conv99=52

Run 3:
  Uniform | feas=0.000 | gen0_best=1929 (92.3% of best) | conv90=1 conv95=28 conv99=63
  Biased  | feas=0.980 | gen0_best=1772 (92.7% of best) | conv90=1 conv95=29 conv99=67

Run 4:
  Uniform | feas=0.000 | gen0_best=1943 (93.0% of best) | conv90=1 conv95=9 conv99=126
  Biased  | feas=0.980 | gen0_best=1714 (89.9% of best) | conv90=18 conv95=31 conv99=86

Run 5:
  Uniform | feas=0.000 | gen0_best=1945 (95.8% of best) | conv90=1 conv95=1 conv99=40
  Biased  | feas=0.940 | gen0_best=1736 (93.3% of best) | conv90=1 conv95=4 conv99=107

SUMMARY

In [52]:
def death_penalty_fitness(solution, values, weights, capacities, n, m):
    """
    Death penalty — infeasible solutions get fitness 0.
    Feasible solutions get their true total value.
    No repair, no multiplier to tune.
    """
    if not is_feasible(solution, weights, capacities, n, m):
        return 0
    return sum(values[j] * solution[j] for j in range(n))


def run_ga_death_penalty(n, m, values, weights, capacities,
                          population_size=50,
                          max_offspring=5000,
                          best_known=None,
                          target_pct=0.99,
                          init_method='uniform',
                          p_j_values=None,
                          seed=None,
                          track_every=50):

    if seed is not None:
        random.seed(seed)

    if init_method == 'uniform':
        population = initialize_population(population_size, n)
    else:
        population = initialize_population_biased(
            population_size, n, p_j_values
        )

    feasible_before = sum(
        1 for sol in population
        if is_feasible(sol, weights, capacities, n, m)
    ) / population_size

    fitness_scores = [
        death_penalty_fitness(sol, values, weights, capacities, n, m)
        for sol in population
    ]

    avg_fitness_gen0 = sum(fitness_scores) / population_size
    best_fitness     = max(fitness_scores)
    best_solution    = population[fitness_scores.index(best_fitness)].copy()
    pct_of_best_gen0 = best_fitness / best_known * 100 if best_known else None

    curve          = [(0, best_fitness, avg_fitness_gen0)]
    convergence_99 = None
    convergence_95 = None
    convergence_90 = None
    offspring_count = 0

    while offspring_count < max_offspring:

        parent1 = tournament_select(population, fitness_scores)
        parent2 = tournament_select(population, fitness_scores)
        child   = uniform_crossover(parent1, parent2, n)
        child   = mutate(child, n)

        child_fitness = death_penalty_fitness(
            child, values, weights, capacities, n, m
        )

        worst_idx = fitness_scores.index(min(fitness_scores))
        if child_fitness > fitness_scores[worst_idx]:
            population[worst_idx]     = child
            fitness_scores[worst_idx] = child_fitness

        if child_fitness > best_fitness:
            best_fitness  = child_fitness
            best_solution = child.copy()

        offspring_count += 1

        if best_known:
            if convergence_99 is None and best_fitness >= best_known * target_pct:
                convergence_99 = offspring_count
            if convergence_95 is None and best_fitness >= best_known * 0.95:
                convergence_95 = offspring_count
            if convergence_90 is None and best_fitness >= best_known * 0.90:
                convergence_90 = offspring_count

        if offspring_count % track_every == 0:
            avg_fit = sum(fitness_scores) / population_size
            curve.append((offspring_count, best_fitness, avg_fit))

    return {
        'feasibility_rate_gen0': feasible_before,
        'avg_fitness_gen0':      avg_fitness_gen0,
        'pct_of_best_gen0':      pct_of_best_gen0,
        'best_fitness_final':    best_fitness,
        'pct_of_best_final':     best_fitness / best_known * 100 if best_known else None,
        'convergence_90':        convergence_90,
        'convergence_95':        convergence_95,
        'convergence_99':        convergence_99,
        'curve':                 curve
    }


def clean_baseline_death_penalty(values, weights, capacities,
                                   p_j_values, reference_best,
                                   n=100, m=3,
                                   num_runs=5,
                                   population_size=50,
                                   max_offspring=5000):

    print("=" * 65)
    print("CLEAN BASELINE — death penalty, no repair operator")
    print(f"n={n}, m={m}, tightness=0.25")
    print(f"Runs: {num_runs} | Pop: {population_size} | "
          f"Max offspring: {max_offspring}")
    print(f"Reference best: {reference_best}")
    print("=" * 65 + "\n")

    u_results = []
    b_results = []

    for run in range(num_runs):

        u = run_ga_death_penalty(
            n, m, values, weights, capacities,
            population_size=population_size,
            max_offspring=max_offspring,
            best_known=reference_best,
            init_method='uniform',
            seed=run * 10,
            track_every=50
        )
        u_results.append(u)

        b = run_ga_death_penalty(
            n, m, values, weights, capacities,
            population_size=population_size,
            max_offspring=max_offspring,
            best_known=reference_best,
            init_method='biased',
            p_j_values=p_j_values,
            seed=run * 10,
            track_every=50
        )
        b_results.append(b)

        print(f"Run {run+1}: "
              f"Uniform  feas={u['feasibility_rate_gen0']:.3f} "
              f"gen0={u['pct_of_best_gen0']:.1f}% "
              f"conv90={u['convergence_90']} "
              f"conv99={u['convergence_99']}"
              f"  |  "
              f"Biased   feas={b['feasibility_rate_gen0']:.3f} "
              f"gen0={b['pct_of_best_gen0']:.1f}% "
              f"conv90={b['convergence_90']} "
              f"conv99={b['convergence_99']}")

    def avg(lst, key):
        vals = [x[key] for x in lst if x[key] is not None]
        return sum(vals) / len(vals) if vals else None

    def pct_none(lst, key):
        return sum(1 for x in lst if x[key] is None) / len(lst) * 100

    print("\n" + "=" * 65)
    print("SUMMARY — death penalty baseline")
    print("=" * 65)
    print(f"\n{'Metric':<42} {'Uniform':>10} {'Biased':>10}")
    print("-" * 65)

    rows = [
        ('Feasibility rate at gen 0',        'feasibility_rate_gen0', '.3f'),
        ('Avg fitness at gen 0 (% of best)', 'pct_of_best_gen0',      '.1f'),
        ('Final fitness (% of best)',         'pct_of_best_final',     '.1f'),
    ]
    for label, key, fmt in rows:
        u_val = avg(u_results, key)
        b_val = avg(b_results, key)
        suffix = '%' if 'pct' in key else ''
        print(f"{label:<42} "
              f"{format(u_val, fmt)+suffix:>10} "
              f"{format(b_val, fmt)+suffix:>10}")

    print()
    for label, key in [('Offspring to reach 90%', 'convergence_90'),
                        ('Offspring to reach 95%', 'convergence_95'),
                        ('Offspring to reach 99%', 'convergence_99')]:
        u_val = avg(u_results, key)
        b_val = avg(b_results, key)
        u_str = f"{u_val:.0f}" if u_val else "never"
        b_str = f"{b_val:.0f}" if b_val else "never"
        print(f"{label:<42} {u_str:>10} {b_str:>10}")
        if u_val and b_val:
            imp = (u_val - b_val) / u_val * 100
            print(f"{'  improvement':<42} {'':>10} {imp:>+9.1f}%")

    print()
    print(f"{'Never reached 99% uniform':<42} "
          f"{pct_none(u_results,'convergence_99'):>9.0f}%")
    print(f"{'Never reached 99% biased':<42} "
          f"{pct_none(b_results,'convergence_99'):>9.0f}%")

    # ── results log for later documentation ─────────────────
    print("\n" + "=" * 65)
    print("RESULTS LOG (save this for documentation)")
    print("=" * 65)
    print(f"Experiment: Clean baseline, death penalty, no repair")
    print(f"Instance:   Random MKP, n={n}, m={m}, α=0.25, seed=42")
    print(f"p_j method: GPU Monte Carlo, 10M samples, floor=0.05")
    print(f"Runs:       {num_runs}, population={population_size}, "
          f"max_offspring={max_offspring}")
    print()
    print(f"Feasibility @ gen 0:  "
          f"Uniform={avg(u_results,'feasibility_rate_gen0'):.1%}  "
          f"Biased={avg(b_results,'feasibility_rate_gen0'):.1%}")
    print(f"Fitness @ gen 0:      "
          f"Uniform={avg(u_results,'pct_of_best_gen0'):.1f}%  "
          f"Biased={avg(b_results,'pct_of_best_gen0'):.1f}%")
    print(f"Conv to 99%:          "
          f"Uniform={avg(u_results,'convergence_99') or 'never'}  "
          f"Biased={avg(b_results,'convergence_99') or 'never'}")
    print(f"Final quality:        "
          f"Uniform={avg(u_results,'pct_of_best_final'):.1f}%  "
          f"Biased={avg(b_results,'pct_of_best_final'):.1f}%")

    return u_results, b_results


# ── run it ───────────────────────────────────────────────────
u_dp, b_dp = clean_baseline_death_penalty(
    values, weights, capacities,
    p_j_values=p_j_values,
    reference_best=reference_best,
    n=100, m=3,
    num_runs=15,
    population_size=50,
    max_offspring=15000
)

CLEAN BASELINE — death penalty, no repair operator
n=100, m=3, tightness=0.25
Runs: 15 | Pop: 50 | Max offspring: 15000
Reference best: 2358

Run 1: Uniform  feas=0.000 gen0=0.0% conv90=None conv99=None  |  Biased   feas=1.000 gen0=47.7% conv90=1836 conv99=None
Run 2: Uniform  feas=0.000 gen0=0.0% conv90=None conv99=None  |  Biased   feas=1.000 gen0=50.1% conv90=1386 conv99=None
Run 3: Uniform  feas=0.000 gen0=0.0% conv90=None conv99=None  |  Biased   feas=0.980 gen0=47.2% conv90=2030 conv99=None
Run 4: Uniform  feas=0.000 gen0=0.0% conv90=None conv99=None  |  Biased   feas=0.980 gen0=44.0% conv90=1152 conv99=None
Run 5: Uniform  feas=0.000 gen0=0.0% conv90=None conv99=None  |  Biased   feas=0.940 gen0=45.9% conv90=1739 conv99=None
Run 6: Uniform  feas=0.000 gen0=0.0% conv90=None conv99=None  |  Biased   feas=1.000 gen0=47.5% conv90=976 conv99=None
Run 7: Uniform  feas=0.000 gen0=0.0% conv90=None conv99=None  |  Biased   feas=1.000 gen0=49.5% conv90=2630 conv99=None
Run 8: Uniform  fea

In [50]:
values_05, weights_05, capacities_05 = generate_random_mkp_instance(
    n=100, m=3, tightness=0.50, seed=42
)

print("Computing p_j for tightness=0.50 instance...")
p_j_data_05 = compute_pj_gpu(
    weights_05, capacities_05,
    n=100, m=3,
    num_samples=1_000_000
)
p_j_values_05 = [r['p_j'] for r in p_j_data_05]

print("Finding reference best...")
ref_05 = run_ga(
    100, 3, values_05, weights_05, capacities_05,
    population_size=100,
    max_offspring=50000,
    best_known=None,
    init_method='uniform',
    seed=999
)['best_fitness']
print(f"Reference best: {ref_05}")

u_05, b_05 = clean_baseline_death_penalty(
    values_05, weights_05, capacities_05,
    p_j_values=p_j_values_05,
    reference_best=ref_05,
    n=100, m=3,
    num_runs=15,
    population_size=50,
    max_offspring=10000
)

Computing p_j for tightness=0.50 instance...
Device: cuda
Samples per item: 1,000,000
Total samples: 100,000,000

Item  10/100 | rho_in=0.29906900 | rho_out=0.34758700 | p_j=0.4625 | elapsed=0.1s
Item  20/100 | rho_in=0.28405200 | rho_out=0.36403400 | p_j=0.4383 | elapsed=0.2s
Item  30/100 | rho_in=0.28749400 | rho_out=0.36054800 | p_j=0.4436 | elapsed=0.2s
Item  40/100 | rho_in=0.28473800 | rho_out=0.36329600 | p_j=0.4394 | elapsed=0.3s
Item  50/100 | rho_in=0.30618100 | rho_out=0.34053300 | p_j=0.4734 | elapsed=0.3s
Item  60/100 | rho_in=0.27762800 | rho_out=0.36986600 | p_j=0.4288 | elapsed=0.4s
Item  70/100 | rho_in=0.29397900 | rho_out=0.35187700 | p_j=0.4552 | elapsed=0.5s
Item  80/100 | rho_in=0.29066900 | rho_out=0.35744500 | p_j=0.4485 | elapsed=0.5s
Item  90/100 | rho_in=0.29784700 | rho_out=0.34943600 | p_j=0.4601 | elapsed=0.6s
Item 100/100 | rho_in=0.27905900 | rho_out=0.36789200 | p_j=0.4313 | elapsed=0.6s

DONE in 0.6s
Items with nonzero rho_in: 100/100
p_j range: [0.413

In [49]:
values_075, weights_075, capacities_075 = generate_random_mkp_instance(
    n=100, m=3, tightness=0.75, seed=42
)

p_j_data_075 = compute_pj_gpu(
    weights_075, capacities_075,
    n=100, m=3, num_samples=500_000
)
p_j_values_075 = [r['p_j'] for r in p_j_data_075]

ref_075 = run_ga(
    100, 3, values_075, weights_075, capacities_075,
    population_size=100, max_offspring=50000,
    best_known=None, init_method='uniform', seed=999
)['best_fitness']
print(f"Reference best: {ref_075}")

u_075, b_075 = clean_baseline_death_penalty(
    values_075, weights_075, capacities_075,
    p_j_values=p_j_values_075,
    reference_best=ref_075,
    n=100, m=3, num_runs=15,
    population_size=50, max_offspring=10000
)

Device: cuda
Samples per item: 500,000
Total samples: 50,000,000

Item  10/100 | rho_in=0.99998600 | rho_out=0.99999400 | p_j=0.5000 | elapsed=0.0s
Item  20/100 | rho_in=0.99999600 | rho_out=0.99999800 | p_j=0.5000 | elapsed=0.1s
Item  30/100 | rho_in=0.99998200 | rho_out=0.99999400 | p_j=0.5000 | elapsed=0.1s
Item  40/100 | rho_in=0.99998000 | rho_out=0.99998800 | p_j=0.5000 | elapsed=0.2s
Item  50/100 | rho_in=0.99998600 | rho_out=0.99998600 | p_j=0.5000 | elapsed=0.2s
Item  60/100 | rho_in=0.99999200 | rho_out=0.99999800 | p_j=0.5000 | elapsed=0.2s
Item  70/100 | rho_in=0.99997200 | rho_out=0.99998400 | p_j=0.5000 | elapsed=0.3s
Item  80/100 | rho_in=0.99998800 | rho_out=0.99999600 | p_j=0.5000 | elapsed=0.3s
Item  90/100 | rho_in=0.99996800 | rho_out=0.99998200 | p_j=0.5000 | elapsed=0.3s
Item 100/100 | rho_in=0.99998400 | rho_out=0.99999800 | p_j=0.5000 | elapsed=0.4s

DONE in 0.4s
Items with nonzero rho_in: 100/100
p_j range: [0.5000, 0.5000]
p_j mean:  0.5000
Reference best: 457

In [38]:
def run_ga_penalty_fitness(n, m, values, weights, capacities,
                            population_size=50,
                            max_offspring=5000,
                            best_known=None,
                            target_pct=0.99,
                            init_method='uniform',
                            p_j_values=None,
                            penalty_multiplier=5.0,
                            seed=None,
                            track_every=50):

    if seed is not None:
        random.seed(seed)

    if init_method == 'uniform':
        population = initialize_population(population_size, n)
    else:
        population = initialize_population_biased(
            population_size, n, p_j_values
        )

    feasible_before = sum(
        1 for sol in population
        if is_feasible(sol, weights, capacities, n, m)
    ) / population_size

    def pfitness(sol):
        total_value = sum(values[j] * sol[j] for j in range(n))
        penalty = 0
        for i in range(m):
            total_weight = sum(weights[j][i] * sol[j] for j in range(n))
            violation = max(0, total_weight - capacities[i])
            penalty += penalty_multiplier * violation
        return total_value - penalty

    fitness_scores   = [pfitness(sol) for sol in population]
    avg_fitness_gen0 = sum(fitness_scores) / population_size
    best_fitness     = max(fitness_scores)
    best_solution    = population[fitness_scores.index(best_fitness)].copy()
    pct_of_best_gen0 = (best_fitness / best_known * 100
                        if best_known and best_fitness > 0 else 0.0)

    curve           = [(0, best_fitness, avg_fitness_gen0)]
    convergence_99  = None
    convergence_95  = None
    convergence_90  = None
    offspring_count = 0

    while offspring_count < max_offspring:

        parent1 = tournament_select(population, fitness_scores)
        parent2 = tournament_select(population, fitness_scores)
        child   = uniform_crossover(parent1, parent2, n)
        child   = mutate(child, n)

        child_fitness = pfitness(child)

        worst_idx = fitness_scores.index(min(fitness_scores))
        if child_fitness > fitness_scores[worst_idx]:
            population[worst_idx]     = child
            fitness_scores[worst_idx] = child_fitness

        if child_fitness > best_fitness:
            best_fitness  = child_fitness
            best_solution = child.copy()

        offspring_count += 1

        # only record convergence if best solution is actually feasible
        if best_known and is_feasible(best_solution, weights,
                                       capacities, n, m):
            if convergence_99 is None and best_fitness >= best_known * target_pct:
                convergence_99 = offspring_count
            if convergence_95 is None and best_fitness >= best_known * 0.95:
                convergence_95 = offspring_count
            if convergence_90 is None and best_fitness >= best_known * 0.90:
                convergence_90 = offspring_count

        if offspring_count % track_every == 0:
            curve.append((offspring_count, best_fitness,
                          sum(fitness_scores)/population_size))

    best_feasible = is_feasible(best_solution, weights, capacities, n, m)

    return {
        'feasibility_rate_gen0':  feasible_before,
        'avg_fitness_gen0':       avg_fitness_gen0,
        'pct_of_best_gen0':       pct_of_best_gen0,
        'best_fitness_final':     best_fitness,
        'best_solution_feasible': best_feasible,
        'pct_of_best_final':      (best_fitness / best_known * 100
                                   if best_known and best_feasible else 0.0),
        'convergence_90':         convergence_90,
        'convergence_95':         convergence_95,
        'convergence_99':         convergence_99,
        'curve':                  curve
    }

In [47]:
u_pf25 = []
b_pf25 = []

for run in range(15):
    u = run_ga_penalty_fitness(
        100, 3, values, weights, capacities,
        population_size=50,
        max_offspring=10000,
        best_known=reference_best,
        init_method='uniform',
        penalty_multiplier=5.0,
        seed=run * 10
    )
    u_pf25.append(u)

    b = run_ga_penalty_fitness(
        100, 3, values, weights, capacities,
        population_size=50,
        max_offspring=10000,
        best_known=reference_best,
        init_method='biased',
        p_j_values=p_j_values,
        penalty_multiplier=5.0,
        seed=run * 10
    )
    b_pf25.append(b)

    print(f"Run {run+1}: "
          f"Uniform  feas={u['feasibility_rate_gen0']:.3f} "
          f"gen0={u['pct_of_best_gen0']:.1f}% "
          f"conv90={u['convergence_90']} "
          f"conv99={u['convergence_99']} "
          f"feasible={u['best_solution_feasible']}"
          f"  |  "
          f"Biased   feas={b['feasibility_rate_gen0']:.3f} "
          f"gen0={b['pct_of_best_gen0']:.1f}% "
          f"conv90={b['convergence_90']} "
          f"conv99={b['convergence_99']} "
          f"feasible={b['best_solution_feasible']}")

def avg(lst, key):
    vals = [x[key] for x in lst if x[key] is not None]
    return sum(vals)/len(vals) if vals else None

def pct_none(lst, key):
    return sum(1 for x in lst if x[key] is None)/len(lst)*100

print("\n" + "="*65)
print("SUMMARY — penalty fitness x5.0, no repair, α=0.25")
print("="*65)
print(f"\n{'Metric':<42} {'Uniform':>10} {'Biased':>10}")
print("-"*65)
print(f"{'Feasibility rate at gen 0':<42} "
      f"{avg(u_pf25,'feasibility_rate_gen0'):>10.3f} "
      f"{avg(b_pf25,'feasibility_rate_gen0'):>10.3f}")
print(f"{'Fitness at gen 0 (% of best)':<42} "
      f"{avg(u_pf25,'pct_of_best_gen0'):>9.1f}% "
      f"{avg(b_pf25,'pct_of_best_gen0'):>9.1f}%")
print(f"{'Final fitness (% of best)':<42} "
      f"{avg(u_pf25,'pct_of_best_final'):>9.1f}% "
      f"{avg(b_pf25,'pct_of_best_final'):>9.1f}%")
print()
for label, key in [('Offspring to 90%','convergence_90'),
                    ('Offspring to 95%','convergence_95'),
                    ('Offspring to 99%','convergence_99')]:
    u_val = avg(u_pf25, key)
    b_val = avg(b_pf25, key)
    u_str = f"{u_val:.0f}" if u_val else "never"
    b_str = f"{b_val:.0f}" if b_val else "never"
    print(f"{label:<42} {u_str:>10} {b_str:>10}")
    if u_val and b_val:
        imp = (u_val - b_val) / u_val * 100
        print(f"{'  improvement':<42} {'':>10} {imp:>+9.1f}%")
print()
print(f"{'Never reached 99% uniform':<42} "
      f"{pct_none(u_pf25,'convergence_99'):>9.0f}%")
print(f"{'Never reached 99% biased':<42} "
      f"{pct_none(b_pf25,'convergence_99'):>9.0f}%")
print("\nRESULTS LOG")
print(f"Experiment: Penalty x5.0, no repair, α=0.25")
print(f"Feasibility @ gen 0: "
      f"Uniform={avg(u_pf25,'feasibility_rate_gen0'):.1%} "
      f"Biased={avg(b_pf25,'feasibility_rate_gen0'):.1%}")
print(f"Fitness @ gen 0:     "
      f"Uniform={avg(u_pf25,'pct_of_best_gen0'):.1f}% "
      f"Biased={avg(b_pf25,'pct_of_best_gen0'):.1f}%")
print(f"Conv to 99%:         "
      f"Uniform={avg(u_pf25,'convergence_99') or 'never'} "
      f"Biased={avg(b_pf25,'convergence_99') or 'never'}")
print(f"Final quality:       "
      f"Uniform={avg(u_pf25,'pct_of_best_final'):.1f}% "
      f"Biased={avg(b_pf25,'pct_of_best_final'):.1f}%")

Run 1: Uniform  feas=0.000 gen0=0.0% conv90=1273 conv99=None feasible=True  |  Biased   feas=1.000 gen0=47.7% conv90=1279 conv99=None feasible=True
Run 2: Uniform  feas=0.000 gen0=0.0% conv90=1291 conv99=None feasible=True  |  Biased   feas=1.000 gen0=50.1% conv90=1260 conv99=None feasible=True
Run 3: Uniform  feas=0.000 gen0=0.0% conv90=1578 conv99=None feasible=True  |  Biased   feas=0.980 gen0=49.0% conv90=1996 conv99=None feasible=True
Run 4: Uniform  feas=0.000 gen0=0.0% conv90=1530 conv99=None feasible=False  |  Biased   feas=0.980 gen0=44.0% conv90=1139 conv99=None feasible=True
Run 5: Uniform  feas=0.000 gen0=0.0% conv90=1493 conv99=None feasible=True  |  Biased   feas=0.940 gen0=45.9% conv90=1270 conv99=None feasible=True
Run 6: Uniform  feas=0.000 gen0=0.0% conv90=2781 conv99=None feasible=False  |  Biased   feas=1.000 gen0=47.5% conv90=1252 conv99=None feasible=True
Run 7: Uniform  feas=0.000 gen0=0.0% conv90=813 conv99=None feasible=True  |  Biased   feas=1.000 gen0=49.5% c

In [48]:
u_pf05 = []
b_pf05 = []

for run in range(15):
    u = run_ga_penalty_fitness(
        100, 3, values_05, weights_05, capacities_05,
        population_size=50,
        max_offspring=10000,
        best_known=ref_05,
        init_method='uniform',
        penalty_multiplier=5.0,
        seed=run * 10
    )
    u_pf05.append(u)

    b = run_ga_penalty_fitness(
        100, 3, values_05, weights_05, capacities_05,
        population_size=50,
        max_offspring=10000,
        best_known=ref_05,
        init_method='biased',
        p_j_values=p_j_values_05,
        penalty_multiplier=5.0,
        seed=run * 10
    )
    b_pf05.append(b)

    print(f"Run {run+1}: "
          f"Uniform  feas={u['feasibility_rate_gen0']:.3f} "
          f"gen0={u['pct_of_best_gen0']:.1f}% "
          f"conv90={u['convergence_90']} "
          f"conv99={u['convergence_99']} "
          f"feasible={u['best_solution_feasible']}"
          f"  |  "
          f"Biased   feas={b['feasibility_rate_gen0']:.3f} "
          f"gen0={b['pct_of_best_gen0']:.1f}% "
          f"conv90={b['convergence_90']} "
          f"conv99={b['convergence_99']} "
          f"feasible={b['best_solution_feasible']}")

def avg(lst, key):
    vals = [x[key] for x in lst if x[key] is not None]
    return sum(vals)/len(vals) if vals else None

def pct_none(lst, key):
    return sum(1 for x in lst if x[key] is None)/len(lst)*100

print("\n" + "="*65)
print("SUMMARY — penalty fitness x5.0, no repair, α=0.50")
print("="*65)
print(f"\n{'Metric':<42} {'Uniform':>10} {'Biased':>10}")
print("-"*65)
print(f"{'Feasibility rate at gen 0':<42} "
      f"{avg(u_pf05,'feasibility_rate_gen0'):>10.3f} "
      f"{avg(b_pf05,'feasibility_rate_gen0'):>10.3f}")
print(f"{'Fitness at gen 0 (% of best)':<42} "
      f"{avg(u_pf05,'pct_of_best_gen0'):>9.1f}% "
      f"{avg(b_pf05,'pct_of_best_gen0'):>9.1f}%")
print(f"{'Final fitness (% of best)':<42} "
      f"{avg(u_pf05,'pct_of_best_final'):>9.1f}% "
      f"{avg(b_pf05,'pct_of_best_final'):>9.1f}%")
print()
for label, key in [('Offspring to 90%','convergence_90'),
                    ('Offspring to 95%','convergence_95'),
                    ('Offspring to 99%','convergence_99')]:
    u_val = avg(u_pf05, key)
    b_val = avg(b_pf05, key)
    u_str = f"{u_val:.0f}" if u_val else "never"
    b_str = f"{b_val:.0f}" if b_val else "never"
    print(f"{label:<42} {u_str:>10} {b_str:>10}")
    if u_val and b_val:
        imp = (u_val - b_val) / u_val * 100
        print(f"{'  improvement':<42} {'':>10} {imp:>+9.1f}%")
print()
print(f"{'Never reached 99% uniform':<42} "
      f"{pct_none(u_pf05,'convergence_99'):>9.0f}%")
print(f"{'Never reached 99% biased':<42} "
      f"{pct_none(b_pf05,'convergence_99'):>9.0f}%")
print("\nRESULTS LOG")
print(f"Experiment: Penalty x5.0, no repair, α=0.50")
print(f"Feasibility @ gen 0: "
      f"Uniform={avg(u_pf05,'feasibility_rate_gen0'):.1%} "
      f"Biased={avg(b_pf05,'feasibility_rate_gen0'):.1%}")
print(f"Fitness @ gen 0:     "
      f"Uniform={avg(u_pf05,'pct_of_best_gen0'):.1f}% "
      f"Biased={avg(b_pf05,'pct_of_best_gen0'):.1f}%")
print(f"Conv to 99%:         "
      f"Uniform={avg(u_pf05,'convergence_99') or 'never'} "
      f"Biased={avg(b_pf05,'convergence_99') or 'never'}")
print(f"Final quality:       "
      f"Uniform={avg(u_pf05,'pct_of_best_final'):.1f}% "
      f"Biased={avg(b_pf05,'pct_of_best_final'):.1f}%")

Run 1: Uniform  feas=0.260 gen0=66.4% conv90=641 conv99=None feasible=True  |  Biased   feas=0.760 gen0=69.9% conv90=523 conv99=9205 feasible=True
Run 2: Uniform  feas=0.220 gen0=70.2% conv90=1093 conv99=None feasible=True  |  Biased   feas=0.720 gen0=66.8% conv90=481 conv99=None feasible=True
Run 3: Uniform  feas=0.440 gen0=67.2% conv90=576 conv99=1634 feasible=True  |  Biased   feas=0.820 gen0=66.4% conv90=572 conv99=None feasible=True
Run 4: Uniform  feas=0.380 gen0=66.7% conv90=587 conv99=None feasible=True  |  Biased   feas=0.660 gen0=68.2% conv90=504 conv99=None feasible=True
Run 5: Uniform  feas=0.240 gen0=66.6% conv90=761 conv99=None feasible=True  |  Biased   feas=0.640 gen0=69.2% conv90=717 conv99=None feasible=True
Run 6: Uniform  feas=0.320 gen0=67.7% conv90=826 conv99=None feasible=True  |  Biased   feas=0.740 gen0=72.7% conv90=713 conv99=None feasible=True
Run 7: Uniform  feas=0.420 gen0=67.6% conv90=778 conv99=2705 feasible=True  |  Biased   feas=0.740 gen0=66.9% conv90=